# Baseline: нейросетевой синтез многоплоскостного ДОЭ

Перед запуском прикрепите Kaggle Dataset, созданный `build_kaggle_dataset.ipynb`. Notebook обучает зафиксированный baseline: U-Net `4×128×128 → 1×128×128`, фазовый выход `π·tanh`, четыре расчёта Френеля и loss из КК + `1.1 ×` SSIM.

In [ ]:
import csv
import json
import math
import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    from pytorch_msssim import ssim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
    from pytorch_msssim import ssim

# Change this only when your Kaggle Dataset has a different slug.
KAGGLE_DATASET_DIR = Path('/kaggle/input/ai-phonetics-data-v1')
WORK_DIR = Path('/kaggle/working/baseline_run')
IMAGE_SIZE = 128
WAVELENGTH = 532e-9
PIXEL_PITCH = 10e-6
ZS = (24.1e-3, 25.3e-3, 26.5e-3, 27.7e-3)
SEED = 42
BATCH_SIZE = 16
EPOCHS = 40
WARMUP_EPOCHS = 2
PATIENCE = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = min(4, os.cpu_count() or 1)

def find_dataset_root(path: Path) -> Path:
    if (path / 'dataset_metadata.json').is_file():
        return path
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Cannot find exactly one dataset_metadata.json below {path}')
    return matches[0].parent

DATA_ROOT = find_dataset_root(KAGGLE_DATASET_DIR)
BSDS_ROOT = DATA_ROOT / 'BSDS500' / 'data'
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)
print('device =', 'cuda' if torch.cuda.is_available() else 'cpu')


## Загрузчик манифестов

Синтетические каналы восстанавливаются из seed, а натуральные — из исходного JPEG и координат кропа. Каждый канал нормируется на собственный максимум, как требует уточнение организаторов.

In [ ]:
def normalise(image, eps=1e-8):
    maximum = image.amax()
    if maximum <= eps:
        raise ValueError('All-black image cannot be normalized')
    return image / maximum

def to_tensor(image):
    return normalise(torch.from_numpy(np.asarray(image.convert('L'), dtype=np.float32) / 255.0))

def draw_glyph(canvas, rng, value):
    char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]
    font_size = int(rng.integers(22, 66))
    try:
        font = ImageFont.truetype('DejaVuSans.ttf', size=font_size)
    except OSError:
        font = ImageFont.load_default()
    glyph = Image.new('L', (font_size * 2, font_size * 2), 0)
    glyph_draw = ImageDraw.Draw(glyph)
    box = glyph_draw.textbbox((0, 0), char, font=font)
    glyph_draw.text((-box[0], -box[1]), char, fill=value, font=font)
    box = glyph.getbbox()
    if box is not None:
        glyph = glyph.crop(box).rotate(float(rng.uniform(-25, 25)), resample=Image.Resampling.BILINEAR, expand=True)
        canvas.paste(glyph, (int(rng.integers(0, max(1, 129 - glyph.width))), int(rng.integers(0, max(1, 129 - glyph.height)))), glyph)

def synthetic(seed):
    rng = np.random.default_rng(int(seed))
    canvas = Image.new('L', (128, 128), 0)
    draw = ImageDraw.Draw(canvas)
    for _ in range(int(rng.integers(1, 5))):
        kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'point', 'glyph']))
        value, width = int(rng.integers(80, 256)), int(rng.integers(1, 7))
        x0, x1 = sorted(rng.integers(4, 124, size=2).tolist()); y0, y1 = sorted(rng.integers(4, 124, size=2).tolist())
        x1, y1 = min(127, max(x1, x0 + 8)), min(127, max(y1, y0 + 8))
        if kind == 'ellipse': draw.ellipse((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'rectangle': draw.rectangle((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'line': draw.line((x0, y0, x1, y1), fill=value, width=width)
        elif kind == 'ring': draw.ellipse((x0, y0, x1, y1), outline=value, width=width)
        elif kind == 'point':
            radius = int(rng.integers(2, 10)); cx, cy = int(rng.integers(radius, 128 - radius)), int(rng.integers(radius, 128 - radius))
            draw.ellipse((cx - radius, cy - radius, cx + radius, cy + radius), fill=value)
        else: draw_glyph(canvas, rng, value)
    return to_tensor(canvas)

class QuadDataset(Dataset):
    def __init__(self, manifest):
        self.records = [json.loads(line) for line in Path(manifest).read_text(encoding='utf-8').splitlines() if line]
    def __len__(self): return len(self.records)
    def __getitem__(self, index):
        images = []
        for item in self.records[index]['items']:
            if item['source'] == 'synthetic':
                images.append(synthetic(item['seed']))
            else:
                path = BSDS_ROOT / item['path']
                with Image.open(path) as source:
                    x, y = item['crop']['x'], item['crop']['y']
                    images.append(to_tensor(source.crop((x, y, x + 128, y + 128))))
        quad = torch.stack(images)
        assert quad.shape == (4, 128, 128)
        return quad

train_set = QuadDataset(DATA_ROOT / 'manifests' / 'train.jsonl')
val_set = QuadDataset(DATA_ROOT / 'manifests' / 'val.jsonl')
print(f'train={len(train_set)}, val={len(val_set)}, example={tuple(train_set[0].shape)}')


## Baseline U-Net, слой Френеля и loss

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(), nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())

class UpBlock(nn.Module):
    def __init__(self, cin, skip, cout):
        super().__init__(); self.reduce = nn.Conv2d(cin, cout, 1); self.block = DoubleConv(cout + skip, cout)
    def forward(self, x, skip):
        x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.block(torch.cat((self.reduce(x), skip), dim=1))

class BaselineUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e0, self.e1, self.e2, self.e3 = DoubleConv(4, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256)
        self.pool, self.b = nn.MaxPool2d(2), DoubleConv(256, 512)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        e0 = self.e0(x); e1 = self.e1(self.pool(e0)); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        x = self.b(self.pool(e3)); x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return math.pi * torch.tanh(self.head(x))

class FresnelLayer(nn.Module):
    def __init__(self, device):
        super().__init__(); c = (torch.arange(128, dtype=torch.float64) - 64) * PIXEL_PITCH; u, v = torch.meshgrid(c, c, indexing='ij')
        kernels = [torch.exp(1j * math.pi * (u.square() + v.square()) / (WAVELENGTH * z)).to(torch.complex64) for z in ZS]
        self.register_buffer('kernels', torch.stack(kernels).to(device), persistent=False)
    def forward(self, phase):
        field = torch.polar(torch.ones_like(phase, dtype=torch.float32), phase.float())
        outputs = []
        for kernel in self.kernels:
            propagated = torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field * kernel, dim=(-2, -1))), dim=(-2, -1)).abs()
            outputs.append(propagated)
        amplitude = torch.cat(outputs, dim=1)
        return amplitude / (amplitude.amax(dim=(-2, -1), keepdim=True) + 1e-8)

def pearson_cc(a, b, eps=1e-8):
    a = a - a.mean(dim=(-2, -1), keepdim=True); b = b - b.mean(dim=(-2, -1), keepdim=True)
    return (a * b).sum(dim=(-2, -1)) / (torch.sqrt(a.square().sum(dim=(-2, -1)) * b.square().sum(dim=(-2, -1))) + eps)

def doe_loss(target, reconstruction):
    cc = pearson_cc(target, reconstruction)
    ssim_values = torch.stack([ssim(target[:, m:m+1], reconstruction[:, m:m+1], data_range=1.0, size_average=True) for m in range(4)])
    return (1 - cc).mean() + 1.1 * (1 - ssim_values).mean(), cc.mean(), ssim_values.mean()


## Обучение

Физический слой и loss всегда выполняются в `float32`; AMP применяется только к U-Net. Лучший checkpoint выбирается по `val_loss`, а КК и ИСС сохраняются раздельно.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin_memory = device.type == 'cuda'
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0)
model, physics = BaselineUNet().to(device), FresnelLayer(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
def multiplier(epoch):
    if epoch < WARMUP_EPOCHS: return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return (1e-5 / LR) + (1 - 1e-5 / LR) * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

@torch.no_grad()
def validate():
    model.eval(); totals = torch.zeros(3, device=device); count = 0
    for target in val_loader:
        target = target.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            phase = model(2 * target - 1)
        reconstruction = physics(phase.float())
        loss, cc, ssim_value = doe_loss(target, reconstruction)
        totals += torch.stack((loss, cc, ssim_value)) * target.shape[0]; count += target.shape[0]
    return (totals / count).cpu().tolist()

history, best_loss, stale = [], float('inf'), 0
for epoch in range(EPOCHS):
    model.train(); running, seen = 0.0, 0
    for target in train_loader:
        target = target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            phase = model(2 * target - 1)
        with torch.autocast(device_type=device.type, enabled=False):
            loss, _, _ = doe_loss(target, physics(phase.float()))
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        running += loss.detach().item() * target.shape[0]; seen += target.shape[0]
    val_loss, val_cc, val_ssim = validate(); scheduler.step()
    row = {'epoch': epoch + 1, 'train_loss': running / seen, 'val_loss': val_loss, 'val_cc': val_cc, 'val_ssim': val_ssim, 'lr': optimizer.param_groups[0]['lr']}
    history.append(row); print(row)
    torch.save({'epoch': epoch + 1, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'metrics': row}, WORK_DIR / 'last.pt')
    if val_loss < best_loss:
        best_loss, stale = val_loss, 0
        torch.save({'epoch': epoch + 1, 'model': model.state_dict(), 'metrics': row}, WORK_DIR / 'best.pt')
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f'Early stopping after {epoch + 1} epochs'); break

with (WORK_DIR / 'metrics.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=history[0].keys()); writer.writeheader(); writer.writerows(history)
(WORK_DIR / 'config.json').write_text(json.dumps({'image_size': IMAGE_SIZE, 'wavelength': WAVELENGTH, 'pixel_pitch': PIXEL_PITCH, 'zs': ZS, 'batch_size': BATCH_SIZE, 'epochs_requested': EPOCHS, 'seed': SEED}, indent=2))
print('Saved:', WORK_DIR / 'best.pt', WORK_DIR / 'metrics.csv')
